In [38]:
# *** MUST be the very first lines of your script/notebook! ***
import sys, importlib

# Top-level alias
sys.modules['numpy._core'] = importlib.import_module('numpy.core')

# Common sub-modules where ufuncs & arrays live
sys.modules['numpy._core.multiarray']    = importlib.import_module('numpy.core.multiarray')
sys.modules['numpy._core.umath']         = importlib.import_module('numpy.core.umath')
sys.modules['numpy._core._multiarray_umath'] = importlib.import_module('numpy.core._multiarray_umath')


In [48]:
import random
random.seed(42)

In [46]:
import os
import json
import random
import re
import glob
import pandas as pd
import numpy as np
from joblib import load
import shap
from catboost import CatBoostRegressor, CatBoostClassifier
from xgboost import XGBRegressor, XGBClassifier

In [40]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Trying to unpickle estimator.*version 1\\.6\\.1 when using version 1\\.3\\.2"
)

In [ ]:
reg_models = {}
for path in glob.glob("regression/Best Model per Target/*"):
    fname = os.path.basename(path)

    # catboost .cbm
    m = re.match(r"catboost_(?P<t>\w+)\.cbm$", fname)
    if m:
        tgt = m.group("t").capitalize()
        reg_models.setdefault(tgt, {})["regression"] = {
            "model_class": CatBoostRegressor,
            "model_ext": ".cbm",
            "model_path": path
        }
        continue

    m = re.match(r"xgb_model_(?P<t>\w+)\.json$", fname)
    if m:
        raw = m.group("t")
        tgt = "".join(part.capitalize() for part in raw.split("_"))
        reg_models.setdefault(tgt, {})["regression"] = {
            "model_class": XGBRegressor,
            "model_ext": ".json",
            "model_path": path
        }


In [ ]:
clf_models = {}
for path in glob.glob("classification/saved_overall_best_models/*.joblib"):
    fname = os.path.basename(path)
    # Grab the last underscore‐delimited chunk:
    raw = os.path.splitext(fname)[0].split("_")[-1]      # e.g. "HumanReproducibility"
    tgt = raw.lower().capitalize()                       # → "Humanreproducibility"
    
    # Pick the right class
    if "CatBoost" in fname:
        cls = CatBoostClassifier
    elif "XGB" in fname or "XGBoost" in fname:
        cls = XGBClassifier
    else:
        raise ValueError(f"Unknown classifier type in {fname}")
    
    clf_models.setdefault(tgt, {})["classification"] = {
        "model_class": cls,
        "model_ext": ".joblib",
        "model_path": path
    }

In [43]:
target_configs = {}

for tgt, cfg in reg_models.items():
    target_configs[tgt] = cfg.copy()

for tgt, cfg in clf_models.items():
    if tgt in target_configs:
        target_configs[tgt].update(cfg)
    else:
        target_configs[tgt] = cfg.copy()


In [44]:
target_configs

{'Rhythm': {'regression': {'model_class': catboost.core.CatBoostRegressor,
   'model_ext': '.cbm',
   'model_path': 'regression/Best Model per Target/catboost_rhythm.cbm'},
  'classification': {'model_class': xgboost.sklearn.XGBClassifier,
   'model_ext': '.joblib',
   'model_path': 'classification/saved_overall_best_models/OVERALL_BEST_XGBoost_GridCV_Rhythm.joblib'}},
 'Humancharacterization': {'regression': {'model_class': catboost.core.CatBoostRegressor,
   'model_ext': '.cbm',
   'model_path': 'regression/Best Model per Target/catboost_humancharacterization.cbm'},
  'classification': {'model_class': xgboost.sklearn.XGBClassifier,
   'model_ext': '.joblib',
   'model_path': 'classification/saved_overall_best_models/OVERALL_BEST_XGBoost_GridCV_HumanCharacterization.joblib'}},
 'Publicinvolvement': {'regression': {'model_class': xgboost.sklearn.XGBRegressor,
   'model_ext': '.json',
   'model_path': 'regression/Best Model per Target/xgb_model_publicinvolvement.json'},
  'classificatio

In [ ]:
def export_instances_for_target(
    task, tgt, X, preds, shap_vals,
    n_samples=500, base_dir="shap_sample_instances"
):
    """
    Export up to n_samples JSONs for a given task/target.
    Creates: shap_sample_instances/{task}/{tgt}/instance_<i>.json
    """
    
    out_dir = os.path.join(base_dir, task, tgt)
    os.makedirs(out_dir, exist_ok=True)
    total = len(X)
    indices = random.sample(range(total), min(n_samples, total))
    for i in indices:
        inst = {
            "instance_id": int(i),
            "features": X.iloc[i].to_dict(),
            "prediction": float(preds[i]),
            "shap_values": dict(zip(X.columns, shap_vals[i].tolist()))
        }
        with open(os.path.join(out_dir, f"instance_{i}.json"), "w") as f:
            json.dump(inst, f, indent=2)

In [51]:
for task in ["regression", "classification"]:
    print(f"\n=== {task.upper()} ===")
    for tgt, cfgs in target_configs.items():
        cfg = cfgs.get(task)
        if not cfg:
            print(f"[{tgt}] no {task} model found, skipping")
            continue

        # 1) load test set
        if task == "regression":
            x_path = f"data/regression/Data Splits/{tgt}/X_test_{tgt.lower()}.csv"
        else:
            x_path = f"data/classification_models_data/EvaluationChoreography{tgt}/X_holdout.csv"
        print(f"[{tgt}] Loading test set from {x_path}")
        X_test = pd.read_csv(x_path)

        # 2) load model
        ModelClass = cfg["model_class"]
        ext        = cfg["model_ext"]
        m_path     = cfg["model_path"]
        try:
            if ext == ".joblib":
                model = load(m_path)
            elif ext in (".cbm", ".json"):
                model = ModelClass()
                model.load_model(m_path)
            else:
                raise ValueError(f"Unknown extension {ext}")
        except Exception as e:
            print(f"[{tgt}] ERROR loading model ({ext}) at {m_path}:\n  {e}")
            continue

        # 3) predict
        preds = model.predict(X_test)
        print(f"[{tgt}] First 5 preds:", preds[:5])

        # 4) compute / cache SHAP
        raw_cache = f"shap/results/{task}/{tgt}/raw_{tgt}.npy"
        out_cache = f"shap/results/{task}/{tgt}/shap_outputs/{tgt}_shap.npy"
        if os.path.exists(raw_cache):
            shap_vals = np.load(raw_cache)
            print(f"[{tgt}] Loaded SHAP raw array:", shap_vals.shape)
        else:
            explainer  = shap.TreeExplainer(model, X_test, feature_dependence="independent")
            shap_vals  = explainer.shap_values(X_test)
            os.makedirs(os.path.dirname(raw_cache), exist_ok=True)
            np.save(raw_cache, shap_vals)
            print(f"[{tgt}] Computed & saved SHAP raw:", shap_vals.shape)

        # also save the “official” shap_outputs file
        os.makedirs(os.path.dirname(out_cache), exist_ok=True)
        np.save(out_cache, shap_vals)

        # 5) export 500 random instances
        export_instances_for_target(task, tgt, X_test, preds, shap_vals)
        print(f"[{tgt}] Exported up to 500 JSONs to shap_sample_instances/{task}/{tgt}")

        print("*" * 100)



=== REGRESSION ===
[Rhythm] Loading test set from data/regression/Data Splits/Rhythm/X_test_rhythm.csv
[Rhythm] First 5 preds: [3.74075467 3.08670534 3.99864816 3.38735734 3.58150775]
[Rhythm] Loaded SHAP raw array: (857, 25)
[Rhythm] Exported up to 500 JSONs to shap_sample_instances/regression/Rhythm
****************************************************************************************************
[Humancharacterization] Loading test set from data/regression/Data Splits/Humancharacterization/X_test_humancharacterization.csv
[Humancharacterization] First 5 preds: [3.2319726  2.71111425 3.26977775 3.21635116 3.05619995]
[Humancharacterization] Loaded SHAP raw array: (857, 25)
[Humancharacterization] Exported up to 500 JSONs to shap_sample_instances/regression/Humancharacterization
****************************************************************************************************
[Publicinvolvement] Loading test set from data/regression/Data Splits/Publicinvolvement/X_test_publicinvo